### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [4]:

import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [5]:
### Read all the pdf's in the directory
def process_all_pdfs(pdf_directory):
    """process all pdf files in the given directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find ll pd file recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}...")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            # Add metadata about source file and file type
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}")
        except Exception as e:
            print(f"Failed to process {pdf_file.name}: {e}")
    print(f"Loaded {len(all_documents)} pages total")
    return all_documents
# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Ignoring wrong pointing object 11 0 (offset 0)


Found 2 PDF files in ../data
Processing git-cheat-sheet.pdf...
Loaded 2 pages from git-cheat-sheet.pdf
Processing keyboard-shortcuts-windows.pdf...
Loaded 1 pages from keyboard-shortcuts-windows.pdf
Loaded 3 pages total


In [6]:
all_pdf_documents

[Document(metadata={'producer': 'Mac OS X 10.9.1 Quartz PDFContext', 'creator': 'Adobe Illustrator CC (Macintosh)', 'creationdate': "D:20140224195805Z00'00'", 'title': 'git-cheat-sheet-education', 'moddate': "D:20140224195805Z00'00'", 'source': '..\\data\\pdf_files\\git-cheat-sheet.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'git-cheat-sheet.pdf', 'file_type': 'pdf'}, page_content="GIT CHEAT SHEET\nSTAGE & SNAPSHOT\nWorking with snapshots and the Git staging area\ngit status\nshow modiﬁed ﬁles in working directory, staged for your next commit\ngit add [file]\nadd a ﬁle as it looks now to your next commit (stage)\ngit reset [file]\nunstage a ﬁle while retaining the changes in working directory\ngit diff\ndiﬀ of what is changed but not staged\ngit diff --staged\ndiﬀ of what is staged but not yet committed\ngit commit -m “[descriptive message]”\ncommit your staged content as a new commit snapshot\nSETUP\nConﬁguring user information used across all local repositori

In [7]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap, 
        separators=["\n\n", "\n", " ", ""],
        length_function=len
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print("Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}")  # print first 200 characters of the first chunk
        print(f"Metadata: {split_docs[0].metadata}")  # print metadata of the first chunk
    return split_docs   


In [8]:
chunks = split_documents(all_pdf_documents)
chunks

Split 3 documents into 12 chunks
Example chunk:
Content: GIT CHEAT SHEET
STAGE & SNAPSHOT
Working with snapshots and the Git staging area
git status
show modiﬁed ﬁles in working directory, staged for your next commit
git add [file]
add a ﬁle as it looks now
Metadata: {'producer': 'Mac OS X 10.9.1 Quartz PDFContext', 'creator': 'Adobe Illustrator CC (Macintosh)', 'creationdate': "D:20140224195805Z00'00'", 'title': 'git-cheat-sheet-education', 'moddate': "D:20140224195805Z00'00'", 'source': '..\\data\\pdf_files\\git-cheat-sheet.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'git-cheat-sheet.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Mac OS X 10.9.1 Quartz PDFContext', 'creator': 'Adobe Illustrator CC (Macintosh)', 'creationdate': "D:20140224195805Z00'00'", 'title': 'git-cheat-sheet-education', 'moddate': "D:20140224195805Z00'00'", 'source': '..\\data\\pdf_files\\git-cheat-sheet.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'git-cheat-sheet.pdf', 'file_type': 'pdf'}, page_content='GIT CHEAT SHEET\nSTAGE & SNAPSHOT\nWorking with snapshots and the Git staging area\ngit status\nshow modiﬁed ﬁles in working directory, staged for your next commit\ngit add [file]\nadd a ﬁle as it looks now to your next commit (stage)\ngit reset [file]\nunstage a ﬁle while retaining the changes in working directory\ngit diff\ndiﬀ of what is changed but not staged\ngit diff --staged\ndiﬀ of what is staged but not yet committed\ngit commit -m “[descriptive message]”\ncommit your staged content as a new commit snapshot\nSETUP\nConﬁguring user information used across all local repositori

### Embedding And VectorStoreDB


In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class EmbeddingManager:
    """Handles embedding generation using sentence-transformers and manages the ChromaDB collection."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding model and ChromaDB collection.
        Args:
            model_name (str): The name of the sentence-transformers(HuggingFace) model to use for embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence-transformers model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded embedding successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.
        Args:
            texts (List[str]): A list of strings to generate embeddings for.    
        Returns:
            np.ndarray: A 2D numpy array where each row is the embedding of the corresponding text.
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        print (f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        """Get the dimensionality of the embeddings produced by the model."""
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        return self.model.get_sentence_embedding_dimension()    

In [11]:
### initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1298.42it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedding successfully. Embedding dimension: 384


### Vector Store

In [12]:
class VectorStore:
    """Manages a ChromaDB collection for storing and retrieving document chunks and their embeddings."""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        Args:
            collection_name (str): The name of the ChromaDB collection to use.
            persist_directory (str): Directory to persist the vector store.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
           # Create persistent chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or Create collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "Collection of PDF document chunks and their embeddings"})
            print(f"Vector store initialized with collection '{self.collection_name}' at '{self.persist_directory}'")
            print(f"Current number of documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise    

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents to the vector store with their embeddings.
        Args:
            documents : A list of dictionaries, each containing 'content' and 'metadata' keys.
            embeddings (np.ndarray): A numpy array of embeddings corresponding to the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")
        
        #prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"  # generate unique ID for each document
            ids.append(doc_id)
            
            #Prepare metadata
            metadata = dict(doc.metadata)  # start with existing metadata
            metadata['doc_index'] = i  # add index in the current batch
            metadata['content_length'] = len(doc.page_content)  # add content length
            metadatas.append(metadata)

            #Documentcontent
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist())  # convert numpy array to list 

        #Add to ChromaDB collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

In [13]:
vectorstore = VectorStore()
vectorstore

Vector store initialized with collection 'pdf_documents' at '../data/vector_store'
Current number of documents in collection: 12


In [14]:
### Coverting documents to embeddings
texts = [doc.page_content for doc in chunks]

# Generate embeddings for the document chunks
embeddings = embedding_manager.generate_embeddings(texts)

# store in the vector store
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 12 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.79it/s]

Generated embeddings with shape: (12, 384)
Added 12 documents to the vector store.
Total documents in collection after addition: 24


### Retriever Pipeline From VectorStore

In [15]:
class RAGRetriever:
    """Retrieves relevant document chunks from the vector store based on a query."""
    def __init__(self, embedding_manager: EmbeddingManager, vector_store: VectorStore):
        """Initialize the retriever with an embedding manager and vector store.
        Args:
            embedding_manager (EmbeddingManager): An instance of the EmbeddingManager class.
            vector_store (VectorStore): An instance of the VectorStore class.
        """
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]: 
        """Retrieve relevant document chunks based on the query.
        Args:
            query (str): The input query string.
            top_k (int): The number of top relevant documents to retrieve.
            score_threshold (float): The minimum similarity score for a document to be included in the results.
        Returns:
            List[Dict[str, Any]]: A list of dictionaries containing the retrieved documents and their metadata.
        """
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")

        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]  # get the embedding

        #Search in the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],  # convert to list for ChromaDB
                n_results=top_k
            )
                # Process results
            retrieved_docs = []
            if results['documents'] and results['documents'][0]: 
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]  # cosine distances
                ids = results['ids'][0]
                for i, (doc_id,document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance  # convert distance to similarity
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1  # rank starts at 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents that meet the score threshold.")
            else:
                print("No documents retrieved from the vector store.")  
            return retrieved_docs          
        except Exception as e:
            print(f"Error querying vector store: {e}")
            return []

rag_retriever = RAGRetriever(embedding_manager, vectorstore)
    

In [16]:
rag_retriever

In [21]:
rag_retriever.retrieve("commands for integrated terminal ctrl+`")

Retrieving documents for query: 'commands for integrated terminal ctrl+`' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 59.48it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents that meet the score threshold.


[{'id': 'doc_4e2a3764_11',
  'content': 'Ctrl+K Ctrl+I Show hover \nIntegrated terminal \nCtrl+` Show integrated terminal \nCtrl+Shift+` Create new terminal \nCtrl+C Copy selection \nCtrl+V Paste into active terminal \nCtrl+↑ / ↓ Scroll up/down \nShift+PgUp / PgDn Scroll page up/down \nCtrl+Home / End Scroll to top/bottom \n \nKeyboard shortcuts for Windows \nOther operating systems’ keyboard shortcuts and additional \nunassigned shortcuts available at aka.ms/vscodekeybindings',
  'metadata': {'source_file': 'keyboard-shortcuts-windows.pdf',
   'creator': 'Microsoft® Word for Microsoft 365',
   'page_label': '1',
   'page': 0,
   'content_length': 428,
   'source': '..\\data\\pdf_files\\keyboard-shortcuts-windows.pdf',
   'author': 'Brad Gashler',
   'doc_index': 11,
   'producer': 'Microsoft® Word for Microsoft 365',
   'moddate': '2021-08-17T08:45:56-07:00',
   'file_type': 'pdf',
   'total_pages': 1,
   'creationdate': '2021-08-17T08:45:56-07:00'},
  'similarity_score': 0.2493352890